In [1]:
import os
import time
import requests
import uuid

import refinitiv.data as rd
import pandas as pd
import yfinance as yf
from openai import OpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv

import fin_db as fdb

#rd.open_session(os.getenv('REFINITIV_API_KEY'))
try:
    client = OpenAI(api_key=os.getenv("OPENAI_KEY"))
except Exception as e:
    raise e

logger = fdb.setup_logger('notebook')

## Getting the Equities set up

In [116]:
ETFs = {
    'SPDR S&P 500 ETF Trust': 'SPY',
    'SPDR Dow Jones Industrial Average ETF Trust': 'DIA',
    'Invesco NASDAQ 100 ETF': 'QQQM.O',
    'Vanguard FTSE 100 ETF': 'VUKE.L',
    'Xtrackers CAC 40 UCITS ETF': 'XCAC.DE',
    'Global X DAX Germany ETF': 'DAX.O',
    'State Street SPDR EURO STOXX 50 ETF': 'FEZ',
}

In [117]:
tickers = []
for _, etf_ric in ETFs.items():
    print(f"Fetching constituents for ETF: {etf_ric}")
    constituents = rd.get_data(
        universe=[etf_ric],
        fields=['TR.FundHoldingRIC'],
        use_field_names_in_headers=True,
        parameters={'Endnum': '5000'}
    )
    cleaned = constituents[('TR.FundHoldingRIC').upper()].dropna().tolist()
    print(f"Found {len(cleaned)} constituents.")
    tickers.extend(cleaned)

Fetching constituents for ETF: SPY


/Users/cedricmckeever/projects/fin_db/.venv/lib/python3.12/site-packages/refinitiv/data/_access_layer/get_data_func.py:57:FutureWarning: Parameter 'use_field_names_in_headers' is deprecated and will be removed in future library version v2.0.
/Users/cedricmckeever/projects/fin_db/.venv/lib/python3.12/site-packages/refinitiv/data/content/fundamental_and_reference/_definition.py:147:FutureWarning: Parameter 'use_field_names_in_headers' is deprecated and will be removed in future library version v2.0.


Found 503 constituents.
Fetching constituents for ETF: DIA


/Users/cedricmckeever/projects/fin_db/.venv/lib/python3.12/site-packages/refinitiv/data/_access_layer/get_data_func.py:57:FutureWarning: Parameter 'use_field_names_in_headers' is deprecated and will be removed in future library version v2.0.
/Users/cedricmckeever/projects/fin_db/.venv/lib/python3.12/site-packages/refinitiv/data/content/fundamental_and_reference/_definition.py:147:FutureWarning: Parameter 'use_field_names_in_headers' is deprecated and will be removed in future library version v2.0.


Found 30 constituents.
Fetching constituents for ETF: QQQM.O


...

Found 41 constituents.
Fetching constituents for ETF: DAX.O


/Users/cedricmckeever/projects/fin_db/.venv/lib/python3.12/site-packages/refinitiv/data/_access_layer/get_data_func.py:57:FutureWarning: Parameter 'use_field_names_in_headers' is deprecated and will be removed in future library version v2.0.
/Users/cedricmckeever/projects/fin_db/.venv/lib/python3.12/site-packages/refinitiv/data/content/fundamental_and_reference/_definition.py:147:FutureWarning: Parameter 'use_field_names_in_headers' is deprecated and will be removed in future library version v2.0.


Found 40 constituents.
Fetching constituents for ETF: FEZ


/Users/cedricmckeever/projects/fin_db/.venv/lib/python3.12/site-packages/refinitiv/data/_access_layer/get_data_func.py:57:FutureWarning: Parameter 'use_field_names_in_headers' is deprecated and will be removed in future library version v2.0.
/Users/cedricmckeever/projects/fin_db/.venv/lib/python3.12/site-packages/refinitiv/data/content/fundamental_and_reference/_definition.py:147:FutureWarning: Parameter 'use_field_names_in_headers' is deprecated and will be removed in future library version v2.0.


Found 51 constituents.


In [118]:
tickers.extend([   # Add tickers relevant to personal use
    'ALLY.K', 'BABA.K', 'CNI.N', 'DHT.N', 'DSV.CO', 'FND.N', 'FRO.N',
    'GNK.N', 'GOGL.O', 'GXO.N', 'HUBG.O', 'ICGA.DE', 'INSW.K', 'KEX.N',
    'LLYVK.OQ', 'LPGAS', 'LPX.N', 'LSTR.O', 'MANH.O', 'NU.N', 'R.N',
    'SAIA.OQ', 'SBLK.O', 'SIRI.O', 'SNOW.K', 'SOLB.BR', 'STZ.N', 'TSM.N',
    'UBSG.S'
])
unique_tickers = list(set(tickers))
unique_tickers.sort()
print(len(unique_tickers))

747


In [119]:
# Get data
desired = [
    'TR.CommonName',
    'TR.ExchangeCode',
    'TR.HQCountryCode',
    'CF_CURR',
    'TR.GICSSubIndustryCode',
]

equities = rd.get_data(
    universe=unique_tickers,
    fields=desired,
    parameters={'Endnum': '5000'}
)

# Identify empty strings as NaN
equities = equities.replace('', pd.NA)

/Users/cedricmckeever/projects/fin_db/.venv/lib/python3.12/site-packages/refinitiv/data/_access_layer/get_data_func.py:99:UserWarning: 'parameters' is not applicable for fields ['CF_CURR']
/Users/cedricmckeever/projects/fin_db/.venv/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/cedricmckeever/projects/fin_db/.venv/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [120]:
# Drop equities with no name or empty string
equities = equities.dropna(subset=['Company Common Name'])

# Display non-unique common names
non_unique_names = equities['Company Common Name'].value_counts()
problems = non_unique_names[non_unique_names > 1].index.tolist()
print(equities[equities['Company Common Name'].isin(problems)])

    Instrument Company Common Name Exchange Code  \
62     ASML.AS     ASML Holding NV           AEX   
63     ASML.OQ     ASML Holding NV           NSM   
284     FOX.OQ            Fox Corp           NSM   
285    FOXA.OQ            Fox Corp           NSM   
311    GOOG.OQ        Alphabet Inc           NSM   
312   GOOGL.OQ        Alphabet Inc           NSM   
509     NWS.OQ           News Corp           NSM   
510    NWSA.OQ           News Corp           NSM   

    Country ISO Code of Headquarters  GICS Sub-Industry Code CF_CURR  
62                                NL                45301010     EUR  
63                                NL                45301010     USD  
284                               US                50201020     USD  
285                               US                50201020     USD  
311                               US                50203010     USD  
312                               US                50203010     USD  
509                               

In [ ]:
# Drop, id rather the ADR
equities= equities[equities['Instrument'] != 'ASML.AS']

# Append to name
class_a = ['FOXA.OQ', 'GOOGL.OQ', 'NWSA.OQ']
for name in class_a:
    equities.loc[equities['Instrument'] == name, 'Company Common Name'] += ' (Class A)'
class_b = ['FOX.OQ', 'NWS.OQ']
for name in class_b:
    equities.loc[equities['Instrument'] == name, 'Company Common Name'] += ' (Class B)'
equities.loc[equities['Instrument'] == 'BRK.B', 'Company Common Name'] += ' (Class C)'
equities.loc[equities['Instrument'] == 'GOOG.OQ', 'Company Common Name'] += ' (Class C)'
# Also, theres not a lot of rows without GICS code
print(equities[equities['GICS Sub-Industry Code'].isna()]['Company Common Name'])

39                                    Alliance Witan PLC
269                             F&C Investment Trust PLC
310                               Golden Ocean Group Ltd
356                 iShares MSCI China UCITS ETF USD Acc
424                Invesco Government & Agency Portfolio
425    State Street Institutional Government Money Ma...
529                   Polar Capital Technology Trust PLC
558                         Pershing Square Holdings Ltd
622               Scottish Mortgage Investment Trust PLC
Name: Company Common Name, dtype: string


In [123]:
# Drop! Some werent equities -> filtering on this ensures equities
equities = equities.dropna(subset=['GICS Sub-Industry Code'])

# After that step, only 2 NAs left, so might as well drop them too
equities = equities.dropna()

In [124]:
EXCHANGE_CODE_TO_NAME = {
    'NYS': 'New York Stock Exchange',
    'NYQ': 'New York Stock Exchange',
    'NSM': 'NASDAQ',
    'NSQ': 'NASDAQ',
    'NMS': 'NASDAQ',
    'BAT': 'BATS Exchange',
    'LSE': 'London Stock Exchange',
    'BRU': 'Euronext Brussels',
    'PAR': 'Euronext Paris',
    'AEX': 'Euronext Amsterdam',
    'GER': 'Deutsche Börse (Frankfurt)',
    'MCE': 'Bolsa de Madrid',
    'MIL': 'Borsa Italiana (Milan)',
    'CPH': 'Nasdaq Copenhagen',
    'TOR': 'Toronto Stock Exchange',
    'HEX': 'Nasdaq Helsinki',
    'VTX': 'SIX Swiss Exchange',
}

firm_exchange = list(zip(
    equities['Instrument'],
    equities["Company Common Name"],
    equities['Exchange Code'].map(EXCHANGE_CODE_TO_NAME)
))
firm_exchange

[('A.N', 'Agilent Technologies Inc', 'New York Stock Exchange'),
 ('AAF.L', 'Airtel Africa PLC', 'London Stock Exchange'),
 ('AAL.L', 'Anglo American PLC', 'London Stock Exchange'),
 ('AAPL.OQ', 'Apple Inc', 'NASDAQ'),
 ('ABBV.N', 'AbbVie Inc', 'New York Stock Exchange'),
 ('ABF.L', 'Associated British Foods PLC', 'London Stock Exchange'),
 ('ABI.BR', 'Anheuser-Busch Inbev SA', 'Euronext Brussels'),
 ('ABNB.OQ', 'Airbnb Inc', 'NASDAQ'),
 ('ABT.N', 'Abbott Laboratories', 'New York Stock Exchange'),
 ('ACCP.PA', 'Accor SA', 'Euronext Paris'),
 ('ACGL.OQ', 'Arch Capital Group Ltd', 'NASDAQ'),
 ('ACN.N', 'Accenture PLC', 'New York Stock Exchange'),
 ('AD.AS', 'Koninklijke Ahold Delhaize NV', 'Euronext Amsterdam'),
 ('ADBE.OQ', 'Adobe Inc', 'NASDAQ'),
 ('ADI.OQ', 'Analog Devices Inc', 'NASDAQ'),
 ('ADM.N', 'Archer-Daniels-Midland Co', 'New York Stock Exchange'),
 ('ADML.L', 'Admiral Group PLC', 'London Stock Exchange'),
 ('ADP.OQ', 'Automatic Data Processing Inc', 'NASDAQ'),
 ('ADSGn.DE', '

In [125]:
def prep_tuple_list(tuples: list[tuple[str, str, str]]) -> str:
    return ",".join(
        [
        f'("{ric}" - "{company}" - "{exchange}")'
         for ric, company, exchange in tuples
        ]
    )

yfinance_tickers = []

# Limit processed at a time to avoid context length issues
max_batch_size = 25
for i in range(0, len(firm_exchange), max_batch_size):
    batch = firm_exchange[i:min(i+max_batch_size, len(firm_exchange))]
    batch_size = len(batch)
    user_content = prep_tuple_list(batch)

    class IntendedResult(BaseModel):
        tickers: list[str] | None = Field(min_length=batch_size, max_length=batch_size)

    system_prompt = (
        "You will be provided a list of tuples representing a refinitiv RIC, "
        "company, and exchange as such: ('RIC' - 'Company Name' - 'Exchange Name'), "
        "('RIC' - 'Company Name' - 'Exchange Name'), ... "
        "\nYou must return a vector of corresponding yahoo finance tickers."
    )

    response = client.chat.completions.parse(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
        response_format=IntendedResult,  # Enforce structured output
    )

    parsed = response.choices[0].message.parsed
    if parsed and parsed.tickers:
        yfinance_tickers.extend(parsed.tickers)
        print(
            f"Added {len(parsed.tickers)} tickers"
            f"({len(yfinance_tickers)/len(firm_exchange):.2%})"
        )
    else:
        print("No tickers returned for this batch.")

Added 25 tickers(3.42%)
Added 25 tickers(6.83%)
Added 25 tickers(10.25%)
Added 25 tickers(13.66%)
Added 25 tickers(17.08%)
Added 25 tickers(20.49%)
Added 25 tickers(23.91%)
Added 25 tickers(27.32%)
Added 25 tickers(30.74%)
Added 25 tickers(34.15%)
Added 25 tickers(37.57%)
Added 25 tickers(40.98%)
Added 25 tickers(44.40%)
Added 25 tickers(47.81%)
Added 25 tickers(51.23%)
Added 25 tickers(54.64%)
Added 25 tickers(58.06%)
Added 25 tickers(61.48%)
Added 25 tickers(64.89%)
Added 25 tickers(68.31%)
Added 25 tickers(71.72%)
Added 25 tickers(75.14%)
Added 25 tickers(78.55%)
Added 25 tickers(81.97%)
Added 25 tickers(85.38%)
Added 25 tickers(88.80%)
Added 25 tickers(92.21%)
Added 25 tickers(95.63%)
Added 25 tickers(99.04%)
Added 7 tickers(100.00%)


In [126]:
equities['yfinance_ticker'] = yfinance_tickers

In [127]:
lseg_prices = rd.get_history(
    universe=equities['Instrument'].tolist(),
    fields=['TR.PriceClose'],
    start='2026-02-01',
    end='2026-02-05',
)

/Users/cedricmckeever/projects/fin_db/.venv/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [128]:
# Make numeric
equities['lseg_valid'] = pd.to_numeric(
    equities['Instrument'].map(lseg_prices.transpose()['2026-02-03']),
    errors='coerce'
)

In [ ]:
# Retrieve price from yfinance to compare
yfin = equities['yfinance_ticker'].to_list()

retrievals = []

# Limit tickers to stay safe
size = 25
for i in range(0, len(yfin), size):
    batch = yfin[i:min(i+size, len(yfin))]
    prices = yf.download(
        tickers=batch,
        start='2026-02-01',
        end='2026-02-05',
    )
    close = prices.xs('Close', axis=1, level=0)
    retrievals.append(close)
    time.sleep(1)  # Be nice to Yahoo's servers

yfin_prices = pd.concat(retrievals, axis=1)

[*********************100%***********************]  25 of 25 completed
[**********************88%*****************      ]  22 of 25 completed$AIRP.PA: possibly delisted; no price data found  (1d 2026-02-01 -> 2026-02-05)
[*********************100%***********************]  25 of 25 completed

1 Failed download:
['AIRP.PA']: possibly delisted; no price data found  (1d 2026-02-01 -> 2026-02-05)
[*****************     36%                       ]  9 of 25 completed$AUTOA.L: possibly delisted; no timezone found
[*******************   40%                       ]  10 of 25 completed$AXA.PA: possibly delisted; no timezone found
[*********************100%***********************]  25 of 25 completed

2 Failed downloads:
['AUTOA.L', 'AXA.PA']: possibly delisted; no timezone found
[*******************   40%                       ]  10 of 25 completed$BLL: possibly delisted; no timezone found
[**********************64%******                 ]  16 of 25 completed$BF.B: possibly delisted; no price dat

In [130]:
equities['yfin_valid'] = equities['yfinance_ticker'].map(
    yfin_prices.transpose()['2026-02-03']
)
equities['distance'] = (equities['lseg_valid'] - equities['yfin_valid']).abs() / equities['lseg_valid']
equities[equities['distance'] > 0.005]

,Instrument,Company Common Name,Exchange Code,Country ISO Code of Headquarters,GICS Sub-Industry Code,CF_CURR,yfinance_ticker,lseg_valid,yfin_valid,distance
140,CFG.N,Citizens Financial Group Inc,NYS,US,40101015,USD,CFG,65.97,65.510002,0.006973


In [5]:
equities = pd.read_csv('equities_saved.csv')

In [6]:
# Attempt to build ticker for those that failed
# =============================================
problems = equities['yfin_valid'].isna()
print(f"Found {problems.sum()} problematic tickers.")

print(
    equities['Instrument']
    .apply(lambda x: x.split('.')[-1] if '.' in x else None)
    .unique()
)

RIC_TO_YAHOO_SUFFIX = {
    '.N': '',       # NYSE
    '.O': '',       # NASDAQ
    '.OQ': '',      # NASDAQ
    '.K': '',       # NYSE (alternate)
    '.A': '',       # NYSE Amex
    '.L': '.L',     # London Stock Exchange
    '.DE': '.DE',   # Frankfurt (Xetra)
    '.AS': '.AS',   # Amsterdam
    '.PA': '.PA',   # Paris
    '.BR': '.BR',   # Brussels
    '.MC': '.MC',   # Madrid
    '.MI': '.MI',   # Milan
    '.CO': '.CO',   # Copenhagen
    '.HE': '.HE',   # Helsinki
    '.S': '.ST',    # Stockholm
    '.SW': '.SW',   # Swiss Exchange
    '.TO': '.TO',   # Toronto
}

equities.loc[problems, 'yfinance_ticker'] = (
    equities.loc[problems, 'Instrument']
    .apply(
        lambda x: x.split('.')[0] + RIC_TO_YAHOO_SUFFIX.get(
                ('.' + x.split('.')[-1]), ''
            )
    )
)

Found 53 problematic tickers.
['N' 'L' 'OQ' 'BR' 'PA' 'AS' 'DE' 'K' 'MC' 'Z' 'MI' 'CO' 'TO' 'O' 'HE' 'S']


In [7]:
# Re-fetch yahpoo prices for corrected tickers
# ==========================================
problems = equities['yfin_valid'].isna()
print(f"Re-fetching {problems.sum()} problematic tickers.")

yfin = equities.loc[problems, 'yfinance_ticker'].to_list()
retrievals = [] 
# Limit tickers to stay safe
size = 25
for i in range(0, len(yfin), size):
    batch = yfin[i:min(i+size, len(yfin))]
    prices = yf.download(
        tickers=batch,
        start='2026-02-01',
        end='2026-02-05',
    )
    close = prices.xs('Close', axis=1, level=0)
    close = close.transpose()['2026-02-03']
    retrievals.append(close)
    time.sleep(1)  # Be nice to Yahoo's servers

yfin_prices = pd.concat(retrievals, axis=0)
equities.loc[problems, 'yfin_valid'] = equities.loc[problems, 'yfinance_ticker'].map(
    yfin_prices
)
equities['distance'] = (equities['lseg_valid'] - equities['yfin_valid']).abs() / equities['lseg_valid']
equities[equities['distance'] > 0.005]
print(equities['yfin_valid'].isna().sum())

Re-fetching 53 problematic tickers.


$HNKG_P.DE: possibly delisted; no price data found  (1d 2026-02-01 -> 2026-02-05)
[                       0%                       ]$BKGH.L: possibly delisted; no price data found  (1d 2026-02-01 -> 2026-02-05)
[********              16%                       ]  4 of 25 completed$DAST.PA: possibly delisted; no price data found  (1d 2026-02-01 -> 2026-02-05)
[**********            20%                       ]  5 of 25 completed$AIRP.PA: possibly delisted; no price data found  (1d 2026-02-01 -> 2026-02-05)
$AXAF.PA: possibly delisted; no price data found  (1d 2026-02-01 -> 2026-02-05)
[************          24%                       ]  6 of 25 completed$EUFI.PA: possibly delisted; no price data found  (1d 2026-02-01 -> 2026-02-05)
[*************         28%                       ]  7 of 25 completed$FOUG.PA: possibly delisted; no price data found  (1d 2026-02-01 -> 2026-02-05)
[***************       32%                       ]  8 of 25 completed$DB1GN.DE: possibly delisted; no price data 

44


Fixed some, but not all :(

In [8]:
# Remaing problems
remaining_problems = equities['yfin_valid'].isna()

# Trying a fix with copilot
SPECIAL_RIC_TO_YAHOO = {
    'AIRP.PA': 'AI.PA',
    'AUTOA.L': 'AUTO.L',
    'AXAF.PA': 'CS.PA',
    'BFb.N': 'BF-B',
    'BKGH.L': 'BKG.L',
    'BNRGn.DE': 'BNRG.DE',
    'BOUY.PA': 'EN.PA',
    'BT.L': 'BT-A.L',
    'CAGR.PA': 'CAF.PA',
    'DANO.PA': 'BN.PA',
    'DAST.PA': 'DSY.PA',
    'DB1Gn.DE': 'DB1.DE',
    'DHLn.DE': 'DHL.DE',
    'ESLX.PA': 'ESI.PA',
    'EUFI.PA': 'URW.AS',
    'FMEG.DE': 'FME.DE',
    'FOUG.PA': 'FOUG.PA',
    'FREG.DE': 'FRE.DE',
    'G1AG.DE': 'G1A.DE',
    'G24n.DE': 'G24.DE',
    'HNKG_p.DE': 'HNR1.DE',
    'HNRGn.DE': 'HNR.DE',
    'HRMS.PA': 'RMS.PA',
    'LEGD.PA': 'LI.PA',
    'LVMH.PA': 'MC.PA',
    'MICP.PA': 'ML.PA',
    'MRCG.DE': 'MRK.DE',
    'MRON.L': 'MRO.L',
    'NDAFI.HE': 'NESTE.HE',
    'ORAN.PA': 'ORA.PA',
    'PERP.PA': 'RI.PA',
    'PRTP.PA': 'UG.PA',
    'PSHG_p.DE': 'PAH3.DE',
    'PUBP.PA': 'PUB.PA',
    'RENA.PA': 'RNO.PA',
    'SASY.PA': 'SAN.PA',
    'SCHN.PA': 'SGO.PA',
    'SGOB.PA': 'GLE.PA',
    'SJP.L': 'SJP.L',
    'SOGN.PA': 'SW.PA',
    'SY1G.DE': 'SY1.DE',
    'TCFP.PA': 'HO.PA',
    'TTEF.PA': 'FP.PA',
    'VOWG_p.DE': 'VOW3.DE',
}

# Apply fixes
equities.loc[remaining_problems, 'yfinance_ticker'] = (
    equities.loc[remaining_problems, 'Instrument'].replace(SPECIAL_RIC_TO_YAHOO)
)

# Again re-fetch yahpoo prices for corrected tickers
# ==========================================
yfin = equities.loc[remaining_problems, 'yfinance_ticker'].to_list()
retrievals = [] 
# Limit tickers to stay safe
size = 25
for i in range(0, len(yfin), size):
    batch = yfin[i:min(i+size, len(yfin))]
    prices = yf.download(
        tickers=batch,
        start='2026-02-01',
        end='2026-02-05',
    )
    close = prices.xs('Close', axis=1, level=0)
    close = close.transpose()['2026-02-03']
    retrievals.append(close)
    time.sleep(1)  # Be nice to Yahoo's servers

yfin_prices = pd.concat(retrievals, axis=0)
equities.loc[remaining_problems, 'yfin_valid'] = equities.loc[remaining_problems, 'yfinance_ticker'].map(
    yfin_prices
)
equities['distance'] = (equities['lseg_valid'] - equities['yfin_valid']).abs() / equities['lseg_valid']
print(equities[equities['distance'] > 0.005])
print(equities['yfin_valid'].isna().sum())


$FOUG.PA: possibly delisted; no price data found  (1d 2026-02-01 -> 2026-02-05)
[***************       32%                       ]  8 of 25 completed$HNR.DE: possibly delisted; no timezone found
[********************* 44%                       ]  11 of 25 completed$URW.AS: possibly delisted; no timezone found
$BNRG.DE: possibly delisted; no timezone found
[**********************92%*******************    ]  23 of 25 completed$ESI.PA: possibly delisted; no timezone found
[*********************100%***********************]  25 of 25 completed

5 Failed downloads:
['FOUG.PA']: possibly delisted; no price data found  (1d 2026-02-01 -> 2026-02-05)
['HNR.DE', 'URW.AS', 'BNRG.DE', 'ESI.PA']: possibly delisted; no timezone found
$SJP.L: possibly delisted; no timezone found
[**********************58%***                    ]  11 of 19 completed$FP.PA: possibly delisted; no timezone found
[**********************89%******************     ]  17 of 19 completed$UG.PA: possibly delisted; no timezone fo

    Instrument           Company Common Name Exchange Code  \
120    CAGR.PA            Credit Agricole SA           PAR   
138      CFG.N  Citizens Financial Group Inc           NYS   
326  HNKG_p.DE           Henkel AG & Co KGaA           GER   
403    LEGD.PA                    Legrand SA           PAR   
477   NDAFI.HE               Nordea Bank Abp           HEX   
590    SCHN.PA         Schneider Electric SE           PAR   
595    SGOB.PA  Compagnie de Saint Gobain SA           PAR   
614    SOGN.PA           Societe Generale SA           PAR   

    Country ISO Code of Headquarters  GICS Sub-Industry Code CF_CURR  \
120                               FR                40101010     EUR   
138                               US                40101015     USD   
326                               DE                30301010     EUR   
403                               FR                20104010     EUR   
477                               FI                40101010     EUR   
590      

In [9]:
# Fix the remaining manually
empties = equities['yfin_valid'].isna()
suspects = equities['distance'] > 0.005
equities[empties | suspects]

,Instrument,Company Common Name,Exchange Code,Country ISO Code of Headquarters,GICS Sub-Industry Code,CF_CURR,yfinance_ticker,lseg_valid,yfin_valid,distance
104,BNRGn.DE,Brenntag SE,GER,DE,20107010,EUR,BNRG.DE,51.820,NaN,NaN
120,CAGR.PA,Credit Agricole SA,PAR,FR,40101010,EUR,CAF.PA,18.745,119.500000,5.375033
138,CFG.N,Citizens Financial Group Inc,NYS,US,40101015,USD,CFG,65.970,65.510002,0.006973
247,ESLX.PA,EssilorLuxottica SA,PAR,FR,35101020,EUR,ESI.PA,255.900,NaN,NaN
251,EUFI.PA,Eurofins Scientific SE,PAR,LU,35203010,EUR,URW.AS,67.880,NaN,NaN
278,FOUG.PA,Eiffage SA,PAR,FR,20103010,EUR,FOUG.PA,127.600,NaN,NaN
326,HNKG_p.DE,Henkel AG & Co KGaA,GER,DE,30301010,EUR,HNR1.DE,75.540,242.800003,2.214191
327,HNRGn.DE,Hannover Rueck SE,GER,DE,40301050,EUR,HNR.DE,242.800,NaN,NaN
403,LEGD.PA,Legrand SA,PAR,FR,20104010,EUR,LI.PA,137.200,31.959999,0.767055
477,NDAFI.HE,Nordea Bank Abp,HEX,FI,40101010,EUR,NESTE.HE,16.905,21.610001,0.278320


In [ ]:
MANUAL_FIX = {
    'BALL.N': 'BALL',
    'BNRGn.DE': 'BNR.DE',
    'CAGR.PA': 'ACA.PA',
    'CFG.N': 'CFG',
    'DOC.N': 'DOC',
    'ESLX.PA': 'EL.PA',
    'EUFI.PA': 'ERF.PA',
    'FOUG.PA': 'FGR.PA',
    'HNKG_p.DE': 'HEN3.DE',
    'HNRGn.DE': 'HNR1.DE',
    'LEGD.PA': 'LR.PA',
    'LLYVK.OQ': 'LLYVK',
    'NDAFI.HE': 'NDA-FI.HE',
    'PRTP.PA': 'KER.PA',
    'SCHN.PA': 'SU.PA',
    'SGOB.PA': 'SGO.PA',
    'SJP.L': 'STJ.L',
    'SOGN.PA': 'GLE.PA',
    'SW.N': 'SW',
    'TTEF.PA': 'TTE.PA',
    'XYZ.N': 'XYZ'
}

equities.loc[empties | suspects, 'yfinance_ticker'] = (
    equities.loc[empties | suspects, 'Instrument'].replace(MANUAL_FIX)
)

Last check ?

In [11]:
# Retrieve price from yfinance to compare
yfin = equities['yfinance_ticker'].to_list()

retrievals = []

# Limit tickers to stay safe
size = 25
for i in range(0, len(yfin), size):
    batch = yfin[i:min(i+size, len(yfin))]
    prices = yf.download(
        tickers=batch,
        start='2026-02-01',
        end='2026-02-05',
    )
    close = prices.xs('Close', axis=1, level=0)
    retrievals.append(close)
    time.sleep(1)  # Be nice to Yahoo's servers

yfin_prices = pd.concat(retrievals, axis=1)

equities['yfin_valid'] = equities['yfinance_ticker'].map(
    yfin_prices.transpose()['2026-02-03']
)
equities['distance'] = (equities['lseg_valid'] - equities['yfin_valid']).abs() / equities['lseg_valid']

[*********************100%***********************]  25 of 25 completed
[*********************100%***********************]  25 of 25 completed
[*********************100%***********************]  25 of 25 completed
[*********************100%***********************]  25 of 25 completed
[*********************100%***********************]  25 of 25 completed
[*********************100%***********************]  25 of 25 completed
[*********************100%***********************]  25 of 25 completed
[*********************100%***********************]  25 of 25 completed
[*********************100%***********************]  25 of 25 completed
[*********************100%***********************]  25 of 25 completed
[********************* 44%                       ]  11 of 25 completed$EVR.L: possibly delisted; no price data found  (1d 2026-02-01 -> 2026-02-05) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  25 of 25 completed

1 Failed downl

In [12]:
# Check last stragglers
empties = equities['yfin_valid'].isna()
suspects = equities['distance'] > 0.005
equities[empties | suspects]

,Instrument,Company Common Name,Exchange Code,Country ISO Code of Headquarters,GICS Sub-Industry Code,CF_CURR,yfinance_ticker,lseg_valid,yfin_valid,distance
138,CFG.N,Citizens Financial Group Inc,NYS,US,40101015,USD,CFG,65.97,65.510002,0.006973
252,EVRE.L,EVRAZ plc,LSE,GB,15104050,GBp,EVR.L,NaN,NaN,NaN
598,SHLG.DE,Siemens Healthineers AG,GER,DE,35101010,EUR,SHL.DE,42.15,41.160564,0.023474
661,TSCO.OQ,Tractor Supply Co,NSM,US,25504040,USD,TSCO,53.47,NaN,NaN
689,VLO.N,Valero Energy Corp,NYS,US,10102030,USD,VLO,192.27,191.101242,0.006079


Small inconstitenties, but data is okay expect one delisted!

In [13]:
# Drop because unlisted
equities = equities[equities['Instrument'] != 'EVRE.L']

equities

,Instrument,Company Common Name,Exchange Code,Country ISO Code of Headquarters,GICS Sub-Industry Code,CF_CURR,yfinance_ticker,lseg_valid,yfin_valid,distance
0,A.N,Agilent Technologies Inc,NYS,US,35203010,USD,A,132.14,132.139999,4.618976e-09
1,AAF.L,Airtel Africa PLC,LSE,GB,50102010,GBp,AAF.L,324.00,324.000000,0.000000e+00
2,AAL.L,Anglo American PLC,LSE,GB,15104020,GBp,AAL.L,3700.00,3700.000000,0.000000e+00
3,AAPL.OQ,Apple Inc,NSM,US,45202030,USD,AAPL,269.48,269.480011,4.076862e-08
4,ABBV.N,AbbVie Inc,NYS,US,35201010,USD,ABBV,225.66,225.660004,1.622844e-08
...,...,...,...,...,...,...,...,...,...,...
727,ZALG.DE,Zalando SE,GER,DE,25504010,EUR,ZAL.DE,21.48,21.480000,2.131116e-08
728,ZBH.N,Zimmer Biomet Holdings Inc,NYS,US,35101010,USD,ZBH,86.18,86.180000,3.541144e-09
729,ZBRA.OQ,Zebra Technologies Corp,NSM,US,45203010,USD,ZBRA,233.16,233.160004,1.570642e-08
730,ZS.OQ,Zscaler Inc,NSM,US,45103020,USD,ZS,188.05,188.050003,1.622844e-08


In [ ]:
# Last fields I want from rd (forgot)
fetch = rd.get_data(
    universe=list(equities['Instrument']),
    fields='TR.ISIN',
    parameters={'Endnum': '5000'}
).replace('', pd.NA)

# Identify empty strings as NaN
fetch_map = fetch.set_index('Instrument').to_dict()['ISIN']
equities['isin'] = equities['Instrument'].map(fetch_map)

In [ ]:
# Rename columns
rename_map = {
    'Instrument': 'ric',
    'Company Common Name': 'name',
    'Exchange Code': 'exchange',
    'Country ISO Code of Headquarters': 'country_hq',
    'GICS Sub-Industry Code': 'gics_code',
    'CF_CURR': 'currency',
    'ISIN': 'isin'
}

equities = equities.rename(columns=rename_map)

In [ ]:
# Add etoro instrumentIds
def get_etoro(code: str):

    url = "https://public-api.etoro.com/api/v1/market-data/search"
    params = {
        "isin": code
    }

    headers = {
        "x-api-key": "insert",
        "x-user-key": "insert",
        "x-request-id": str(uuid.uuid4())
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        data = response.json()
        # Find the exact match in the returned items list
        instrument = next((item for item in data['items'] if item['isin'] == code), None)

        if instrument:
            return instrument['instrumentId']
        else:
            return pd.NA
    else:
        raise Exception(f"API request failed with status code {response.status_code}: {response.text}")


for index, row in equities.iterrows():
    etoro = get_etoro(row['isin'])
    equities.at[index, 'etoro_iid'] = etoro
    time.sleep(1)  # Add a delay between API requests

In [ ]:
equities['code'] = equities['yfinance_ticker'].apply(
    lambda x: x.split('.')[0] if '.' in x else x
)
equities['code'] = equities['code'].apply(
    lambda x: x.replace('-', '')  # Remove dashes for tickers like BF-B
)
equities['code'] = equities['code'].apply(
    lambda x: x[:4] if len(x) > 4 else x
)

equities['instrument_id'] = equities.apply(
    lambda row: fdb.instrument_id(
        asset_class='equity',
        code=row['code'],
        hash_on=row['isin']
    ),
    axis=1
)

equities = equities[[
    'instrument_id', 'name', 'isin', 'yfinance_ticker', 'etoro_iid',
    'ric', 'exchange', 'country_hq', 'currency', 'gics_code'
]]

,instrument_id,name,isin,yfinance_ticker,etoro_ticker,ric,exchange,country_hq,currency,gics_code
0,EQUAXXXUS00846U1016X,Agilent Technologies Inc,US00846U1016,A,NaN,A.N,NYS,US,USD,35203010
1,EQUAAFXGB00BKDRYJ47X,Airtel Africa PLC,GB00BKDRYJ47,AAF.L,NaN,AAF.L,LSE,GB,GBp,50102010
2,EQUAALXGB00BTK05J60X,Anglo American PLC,GB00BTK05J60,AAL.L,NaN,AAL.L,LSE,GB,GBp,15104020
3,EQUAAPLUS0378331005X,Apple Inc,US0378331005,AAPL,AAPL,AAPL.OQ,NSM,US,USD,45202030
4,EQUABBVUS00287Y1091X,AbbVie Inc,US00287Y1091,ABBV,NaN,ABBV.N,NYS,US,USD,35201010
...,...,...,...,...,...,...,...,...,...,...
726,EQUZALXDE000ZAL1111X,Zalando SE,DE000ZAL1111,ZAL.DE,NaN,ZALG.DE,GER,DE,EUR,25504010
727,EQUZBHXUS98956P1021X,Zimmer Biomet Holdings Inc,US98956P1021,ZBH,NaN,ZBH.N,NYS,US,USD,35101010
728,EQUZBRAUS9892071054X,Zebra Technologies Corp,US9892071054,ZBRA,NaN,ZBRA.OQ,NSM,US,USD,45203010
729,EQUZSXXUS98980G1022X,Zscaler Inc,US98980G1022,ZS,NaN,ZS.OQ,NSM,US,USD,45103020


In [3]:
equities.to_csv('equities_final.csv', index=False)

# ETFs

In [2]:
tickers = [
    "CNDX.L",
    "CEBP.DE",
    "STX50EEX.DE",
    "VVSM.DE",
    "IUS3.DE",
    "USCP.DE",
    "EXI2.DE",
    "IDEM.L",
    "IJPE.L",
    "SPY5.L",
    "XDEW.DE",
    "VERX.DE",
    "SPY4.DE",
    "ZPRR.DE",
    "STOXXIEX.DE",
    "SWDA.L",
    "SPY"
]

fields = [
    'TR.CommonName',
    'TR.ExchangeCode',
    'CF_CURR',
    'TR.ISIN'
]

ETFs = rd.get_data(
    universe=tickers,
    fields=fields,
    parameters={'Endnum': '5000'}
)

# Identify empty strings as NaN
ETFs = ETFs.replace('', pd.NA)

ETFs

/Users/cedricmckeever/projects/fin_db/.venv/lib/python3.12/site-packages/refinitiv/data/_access_layer/get_data_func.py:99:UserWarning: 'parameters' is not applicable for fields ['CF_CURR']


,Instrument,Company Common Name,Exchange Code,ISIN,CF_CURR
0,CNDX.L,iShares NASDAQ 100 UCITS ETF USD (Acc),LSE,IE00B53SZB19,USD
1,CEBP.DE,iShares MSCI EMU USD Hedged UCITS ETF (Acc),GER,IE00BWZN1T31,EUR
2,STX50EEX.DE,iShares Core EURO STOXX 50 UCITS ETF (DE),GER,DE0005933956,EUR
3,VVSM.DE,VanEck Semiconductor UCITS ETF USD A,GER,IE00BMC38736,EUR
4,IUS3.DE,iShares S&P SmallCap 600 UCITS ETF USD (Dist),GER,IE00B2QWCY14,EUR
5,USCP.DE,OssiamShillerBarclaysCAPEUSSecValTR UCITS ETF 1CU,GER,LU1079841273,EUR
6,EXI2.DE,iShares Dow Jones Gl Titans 50 UCITS ETF (DE) ...,GER,DE0006289382,EUR
7,IDEM.L,iShares MSCI EM UCITS ETF USD (Dist),LSE,IE00B0M63177,USD
8,IJPE.L,ISHARES V PLC - iShares MSCI Japan EUR Hedged ...,LSE,IE00B42Z5J44,EUR
9,SPY5.L,SPDR S&P 500 UCITS ETF,LSE,IE00B6YX5C33,USD


In [3]:

puller = fdb.LSEGPuller(
    tickers=tickers,
    sdate='1980-01-01',
    edate='2026-03-20',
    batch_size=3
)

hist_etf = puller.histpull(['close', 'totret'])

hist_etf

,date,identifier,source,field,scale,value
0,2010-09-15,CNDX.L,LSEG,close,1.00,102.15
1,2010-09-27,CNDX.L,LSEG,close,1.00,106.4
2,2010-09-28,CNDX.L,LSEG,close,1.00,106.15
3,2010-09-29,CNDX.L,LSEG,close,1.00,106.06
4,2010-09-30,CNDX.L,LSEG,close,1.00,104.99
...,...,...,...,...,...,...
137980,2026-03-16,SPY,LSEG,totret,0.01,1.017681
137981,2026-03-17,SPY,LSEG,totret,0.01,0.263067
137982,2026-03-18,SPY,LSEG,totret,0.01,-1.39537
137983,2026-03-19,SPY,LSEG,totret,0.01,-0.246436


# Currencies

In [30]:
currencies = [
    "DKK=",
    "CHF=",
    "EUR=",
    "GBP=",
    "CAD=",
    "TWD=",
    "JPY=",
    "HKD=",
    "CNY=",
    "AUD=",
    "DKKUSD=R",
    "CHFUSD=R",
    "USDEUR=R",
    "USDGBP=R",
    "CADUSD=R",
    "TWDUSD=R",
    "JPYUSD=R",
    "HKDUSD=R",
    "CNYUSD=R",
    "USDAUD=R",
]

frames = []
# Get history in batches to be nice to Refinitiv's servers (5 years)
for i in range(2, len(currencies)+2, 2):
    batch = currencies[i-2:i]
    hist_cur = rd.get_history(
        universe=batch,
        fields=['TR.BIDPRICE'],
        start='1980-01-01',
        end='2026-03-22',
    )
    print(f"Fetched history for {batch}")
    frames.append(hist_cur)
    time.sleep(5)  # Be nice to Refinitiv's servers



INFO     httpx: HTTP Request: POST http://localhost:9006/api/udf "HTTP/1.1 200 OK"


Fetched history for ['DKK=', 'CHF=']


INFO     httpx: HTTP Request: POST http://localhost:9006/api/udf "HTTP/1.1 200 OK"


Fetched history for ['EUR=', 'GBP=']


INFO     httpx: HTTP Request: POST http://localhost:9006/api/udf "HTTP/1.1 200 OK"


...

Fetched history for ['CADUSD=R', 'TWDUSD=R']


INFO     httpx: HTTP Request: POST http://localhost:9006/api/udf "HTTP/1.1 200 OK"


Fetched history for ['JPYUSD=R', 'HKDUSD=R']


INFO     httpx: HTTP Request: POST http://localhost:9006/api/udf "HTTP/1.1 200 OK"


Fetched history for ['CNYUSD=R', 'USDAUD=R']


In [31]:
total = pd.concat(frames, axis=1)
total.to_csv('currency_history.csv')
total

Bid Price,DKK=,CHF=,EUR=,GBP=,CAD=,TWD=,JPY=,HKD=,CNY=,AUD=,DKKUSD=R,CHFUSD=R,USDEUR=R,USDGBP=R,CADUSD=R,TWDUSD=R,JPYUSD=R,HKDUSD=R,CNYUSD=R,USDAUD=R
Date,,,,,,,,,,,,,,,,,,,,
1979-12-31,5.3742,1.596,1.5081,2.2252,1.1684,<NA>,240.3,4.9525,<NA>,1.1066,<NA>,<NA>,0.6631,0.4494,<NA>,<NA>,<NA>,<NA>,<NA>,0.9037
1980-01-01,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1980-01-02,5.3427,1.58,1.5149,2.2331,1.1658,<NA>,238.45,4.925,<NA>,1.1085,<NA>,<NA>,0.6601,0.4478,<NA>,<NA>,<NA>,<NA>,<NA>,0.9021
1980-01-03,5.3287,1.5722,1.5177,2.2437,1.1703,<NA>,238.35,4.9305,<NA>,1.11,<NA>,<NA>,0.6589,0.4457,<NA>,<NA>,<NA>,<NA>,<NA>,0.9009
1980-01-04,5.3525,1.5802,1.5129,2.2371,1.1687,<NA>,234.8,4.923,<NA>,1.1101,<NA>,<NA>,0.661,0.447,<NA>,<NA>,<NA>,<NA>,<NA>,0.9008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-16,6.4946,0.7875,1.1503,1.3316,1.3686,31.948,159.06,7.8303,6.8955,0.7069,0.154,1.2689,0.8691,0.750582,0.7306,0.031271,0.628654,0.127704,0.144995,1.4134
2026-03-17,6.475,0.7845,1.1538,1.3354,1.3691,31.849,158.98,7.8361,6.8863,0.7102,0.1544,1.2737,0.8665,0.748671,0.7304,0.031369,0.628812,0.12761,0.145193,1.4069
2026-03-18,6.5247,0.7929,1.145,1.3254,1.3732,32.002,159.86,7.8377,6.8726,0.7022,0.1532,1.2601,0.8731,0.754091,0.7282,0.031219,0.625429,0.127566,0.145493,1.4231


In [32]:
sub = total.loc['1990-01-01':'2026-03-22']
sub

Bid Price,DKK=,CHF=,EUR=,GBP=,CAD=,TWD=,JPY=,HKD=,CNY=,AUD=,DKKUSD=R,CHFUSD=R,USDEUR=R,USDGBP=R,CADUSD=R,TWDUSD=R,JPYUSD=R,HKDUSD=R,CNYUSD=R,USDAUD=R
Date,,,,,,,,,,,,,,,,,,,,
1990-01-01,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1990-01-02,6.6575,1.5808,1.1835,1.6089,1.1599,25.97,146.55,7.8105,4.7339,0.7843,0.1502,0.6322,0.8233,0.6194,0.8618,<NA>,0.6819,0.128016,<NA>,1.2731
1990-01-03,6.675,1.579,1.1774,1.6093,1.16,25.97,145.32,7.8125,4.7339,0.781,0.1498,0.633,0.8267,0.62,0.8617,<NA>,0.6877,0.127984,<NA>,1.2763
1990-01-04,6.53,1.5368,1.2063,1.6375,1.161,25.97,143.33,7.813,4.7339,0.7877,0.1531,0.6503,0.8139,0.6147,0.861,<NA>,0.6974,0.127959,<NA>,1.2747
1990-01-05,6.5575,1.5445,1.1981,1.6345,1.1605,25.89,144.5,7.813,4.7339,0.7831,0.1524,0.647,0.8092,0.6101,0.8613,<NA>,0.6916,0.127975,<NA>,1.2715
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-16,6.4946,0.7875,1.1503,1.3316,1.3686,31.948,159.06,7.8303,6.8955,0.7069,0.154,1.2689,0.8691,0.750582,0.7306,0.031271,0.628654,0.127704,0.144995,1.4134
2026-03-17,6.475,0.7845,1.1538,1.3354,1.3691,31.849,158.98,7.8361,6.8863,0.7102,0.1544,1.2737,0.8665,0.748671,0.7304,0.031369,0.628812,0.12761,0.145193,1.4069
2026-03-18,6.5247,0.7929,1.145,1.3254,1.3732,32.002,159.86,7.8377,6.8726,0.7022,0.1532,1.2601,0.8731,0.754091,0.7282,0.031219,0.625429,0.127566,0.145493,1.4231


# Assembling for ingest

In [2]:
ETFs = pd.read_csv('../../data/ETFs.csv')

In [3]:
rename_map = {
    'Instrument': 'LSEG',
    'Company Common Name': 'name',
    'Exchange Code': 'exchange',
    'CF_CURR': 'unit',
}
ETFs = ETFs.rename(columns=rename_map)
ETFs['asset_class'] = 'equity_etf'
ETFs['YAHOO'] = ETFs['LSEG']
ETFs.loc[ETFs['YAHOO'] == 'STX50EEX.DE', 'YAHOO'] = 'EXW1.DE'
ETFs.loc[ETFs['YAHOO'] == 'STOXXIEX.DE', 'YAHOO'] = 'EXSA.DE'
ETFs['internal_ticker'] = ETFs['YAHOO']

ETFs['code'] = ETFs['YAHOO'].apply(
    lambda x: x.split('.')[0] if '.' in x else x
)
ETFs['code'] = ETFs['code'].apply(
    lambda x: x.replace('-', '')  # Remove dashes
)
ETFs['code'] = ETFs['code'].apply(
    lambda x: x[:4] if len(x) > 4 else x
)
ETFs.head()

,LSEG,name,exchange,ISIN,unit,asset_class,YAHOO,internal_ticker,code
0,CNDX.L,iShares NASDAQ 100 UCITS ETF USD (Acc),LSE,IE00B53SZB19,USD,equity_etf,CNDX.L,CNDX.L,CNDX
1,CEBP.DE,iShares MSCI EMU USD Hedged UCITS ETF (Acc),GER,IE00BWZN1T31,EUR,equity_etf,CEBP.DE,CEBP.DE,CEBP
2,STX50EEX.DE,iShares Core EURO STOXX 50 UCITS ETF (DE),GER,DE0005933956,EUR,equity_etf,EXW1.DE,EXW1.DE,EXW1
3,VVSM.DE,VanEck Semiconductor UCITS ETF USD A,GER,IE00BMC38736,EUR,equity_etf,VVSM.DE,VVSM.DE,VVSM
4,IUS3.DE,iShares S&P SmallCap 600 UCITS ETF USD (Dist),GER,IE00B2QWCY14,EUR,equity_etf,IUS3.DE,IUS3.DE,IUS3


In [4]:
ETFs['instrument_id'] = ETFs.apply(
    lambda row: fdb.create_instrument_id(
        asset_class='equity_etf',
        code=row['code'],
        hash_on=row['ISIN']
    ),
    axis=1
)

ETF_instruments = ETFs[[
    'instrument_id', 'name', 'asset_class', 'unit', 'internal_ticker'
]]

In [5]:
fdb.open_session('fin_db_app')
fdb.queries.ingest_instruments(ETF_instruments, commit=True)

INFO     fin_db.session: Database session opened successfully.
INFO     fin_db.queries.execute: Executed write query from write_instruments.sql (17 rows affected).
INFO     fin_db.queries.execute: Transaction committed.


In [6]:
ETF_attr = ETFs[[
    'instrument_id', 'exchange'
]].copy()

ETF_attr = ETF_attr.rename(columns={'exchange': 'value'})
ETF_attr['field'] = 'exchange'
ETF_attr = ETF_attr[['instrument_id', 'field', 'value']]


In [7]:
fdb.queries.ingest_attributes(ETF_attr, commit=True)

INFO     fin_db.queries.execute: Executed write query from write_attributes.sql (17 rows affected).
INFO     fin_db.queries.execute: Transaction committed.


In [8]:
ETF_hist = pd.read_csv('../../data/hist_etf.csv')

In [9]:
ETF_hist = ETF_hist.merge(
    ETFs[['LSEG', 'instrument_id']],
    left_on='identifier',
    right_on='LSEG',
    how='left'
)
ETF_hist.head()

,date,identifier,source,field,scale,value,LSEG,instrument_id
0,2010-09-15,CNDX.L,LSEG,close,1.0,102.15,CNDX.L,ETFCNDXIE00B53SZB19X
1,2010-09-27,CNDX.L,LSEG,close,1.0,106.40,CNDX.L,ETFCNDXIE00B53SZB19X
2,2010-09-28,CNDX.L,LSEG,close,1.0,106.15,CNDX.L,ETFCNDXIE00B53SZB19X
3,2010-09-29,CNDX.L,LSEG,close,1.0,106.06,CNDX.L,ETFCNDXIE00B53SZB19X
4,2010-09-30,CNDX.L,LSEG,close,1.0,104.99,CNDX.L,ETFCNDXIE00B53SZB19X


In [10]:
ETF_hist = ETF_hist.drop(columns=['LSEG', 'identifier']).dropna()
ETF_hist

,date,source,field,scale,value,instrument_id
0,2010-09-15,LSEG,close,1.00,102.150000,ETFCNDXIE00B53SZB19X
1,2010-09-27,LSEG,close,1.00,106.400000,ETFCNDXIE00B53SZB19X
2,2010-09-28,LSEG,close,1.00,106.150000,ETFCNDXIE00B53SZB19X
3,2010-09-29,LSEG,close,1.00,106.060000,ETFCNDXIE00B53SZB19X
4,2010-09-30,LSEG,close,1.00,104.990000,ETFCNDXIE00B53SZB19X
...,...,...,...,...,...,...
137980,2026-03-16,LSEG,totret,0.01,1.017681,ETFSPYXUS78462F1030X
137981,2026-03-17,LSEG,totret,0.01,0.263067,ETFSPYXUS78462F1030X
137982,2026-03-18,LSEG,totret,0.01,-1.395370,ETFSPYXUS78462F1030X
137983,2026-03-19,LSEG,totret,0.01,-0.246436,ETFSPYXUS78462F1030X


In [11]:
ETF_updates = ETF_hist[['instrument_id', 'field']].drop_duplicates()
ETF_updates['source'] = 'YAHOO' # update with yahoo from now on
ETF_updates['frequency'] = 'daily'

In [12]:
fdb.queries.ingest_updates(ETF_updates, commit=True)

INFO     fin_db.queries.execute: Executed write query from write_updates.sql (34 rows affected).
INFO     fin_db.queries.execute: Transaction committed.


In [13]:
fdb.queries.ingest_observations(ETF_hist, commit=True)

INFO     fin_db.queries.execute: Executed write query from write_observations.sql (137977 rows affected).
INFO     fin_db.queries.execute: Transaction committed.
INFO     fin_db.queries.execute: Executed write query from log_updates.sql (0 rows affected).
INFO     fin_db.queries.execute: Transaction committed.


In [14]:
load_dotenv('../../../scripts/.env')

ETF_ids = ETFs[['instrument_id', 'ISIN', 'LSEG', 'YAHOO']].copy()
ETF_ids['ETORO'] = None

toro = fdb.providers.EtoroAPI(
    x_api_key=os.getenv("ETORO_API"),
    x_user_key=os.getenv("ETORO_USER_READ")
)

for row in ETF_ids.itertuples():
    if pd.notna(row.ISIN):
        try:
            etoro_id = toro.search(row.ISIN, field='isin', return_fields=['instrumentId'])['instrumentId']
            ETF_ids.at[row.Index, 'ETORO'] = etoro_id
        except Exception as e:
            print(f"Error occurred while searching for ISIN {row.ISIN}: {e}")
        time.sleep(1)  # Be nice to API servers

ETF_ids

Error occurred while searching for ISIN IE00B2QWCY14: No instrument found for IE00B2QWCY14 in field isin
Error occurred while searching for ISIN LU1079841273: No instrument found for LU1079841273 in field isin
Error occurred while searching for ISIN DE0006289382: No instrument found for DE0006289382 in field isin
Error occurred while searching for ISIN IE00BLNMYC90: No instrument found for IE00BLNMYC90 in field isin
Error occurred while searching for ISIN IE00BKX55S42: No instrument found for IE00BKX55S42 in field isin
Error occurred while searching for ISIN IE00B4YBJ215: No instrument found for IE00B4YBJ215 in field isin


,instrument_id,ISIN,LSEG,YAHOO,ETORO
0,ETFCNDXIE00B53SZB19X,IE00B53SZB19,CNDX.L,CNDX.L,8015
1,ETFCEBPIE00BWZN1T31X,IE00BWZN1T31,CEBP.DE,CEBP.DE,2909
2,ETFEXW1DE0005933956X,DE0005933956,STX50EEX.DE,EXW1.DE,3400
3,ETFVVSMIE00BMC38736X,IE00BMC38736,VVSM.DE,VVSM.DE,2905
4,ETFIUS3IE00B2QWCY14X,IE00B2QWCY14,IUS3.DE,IUS3.DE,None
5,ETFUSCPLU1079841273X,LU1079841273,USCP.DE,USCP.DE,None
6,ETFEXI2DE0006289382X,DE0006289382,EXI2.DE,EXI2.DE,None
7,ETFIDEMIE00B0M63177X,IE00B0M63177,IDEM.L,IDEM.L,3063
8,ETFIJPEIE00B42Z5J44X,IE00B42Z5J44,IJPE.L,IJPE.L,3068
9,ETFSPY5IE00B6YX5C33X,IE00B6YX5C33,SPY5.L,SPY5.L,2864


In [15]:
mapped = dict()
for row in ETF_ids[ETF_ids['ETORO'].isna()].itertuples():
    time.sleep(1)  # Be nice to API servers
    try:
        val = row.LSEG
        if val == 'EXI2.DE':
            val = 'DJGTEEX.DE'
        maybe = toro.search(val, field='internalSymbolFull', return_fields=['instrumentId', 'internalSymbolFull', 'internalInstrumentDisplayName'])
        mapped[row.LSEG] = maybe['instrumentId']
        print(f"Search for {row.LSEG} returned: {maybe}")
    except Exception as e:
        print(f"failed {row.LSEG}")

Search for IUS3.DE returned: {'instrumentId': 10601, 'internalSymbolFull': 'IUS3.DE', 'internalInstrumentDisplayName': 'iShares S&P Small Cap 600 UCITS ETF USD Dist'}
Search for USCP.DE returned: {'instrumentId': 10616, 'internalSymbolFull': 'USCP.DE', 'internalInstrumentDisplayName': 'Ossiam Shiller Barclays Cape US Sector Value TR'}
Search for EXI2.DE returned: {'instrumentId': 10590, 'internalSymbolFull': 'DJGTEEX.DE', 'internalInstrumentDisplayName': 'iShares Dow Jones Global Titans 50 UCITS ETF'}
Search for XDEW.DE returned: {'instrumentId': 10556, 'internalSymbolFull': 'XDEW.DE', 'internalInstrumentDisplayName': 'Xtrackers S&P 500 Equal Weight UCITS ETF'}
Search for VERX.DE returned: {'instrumentId': 10573, 'internalSymbolFull': 'VERX.DE', 'internalInstrumentDisplayName': 'Vanguard FTSE Developed Europe ex UK UCITS ETF'}
Search for SPY4.DE returned: {'instrumentId': 10583, 'internalSymbolFull': 'SPY4.DE', 'internalInstrumentDisplayName': 'SPDR S&P 400 US Mid Cap UCITS ETF'}


In [16]:
ETF_ids['ETORO'] = ETF_ids.apply(
    lambda row: mapped.get(row.LSEG) if pd.isna(row.ETORO) else row.ETORO,
    axis=1
)
ETF_ids['ETORO'] = ETF_ids['ETORO'].astype('str')
ETF_ids_long = ETF_ids.melt(
    id_vars=['instrument_id'],
    value_vars=['ISIN', 'LSEG', 'YAHOO', 'ETORO'],
    var_name='source',
    value_name='ext_id'
)

ETF_ids_long

,instrument_id,source,ext_id
0,ETFCNDXIE00B53SZB19X,ISIN,IE00B53SZB19
1,ETFCEBPIE00BWZN1T31X,ISIN,IE00BWZN1T31
2,ETFEXW1DE0005933956X,ISIN,DE0005933956
3,ETFVVSMIE00BMC38736X,ISIN,IE00BMC38736
4,ETFIUS3IE00B2QWCY14X,ISIN,IE00B2QWCY14
...,...,...,...
63,ETFSPY4IE00B4YBJ215X,ETORO,10583
64,ETFZPRRIE00BJ38QD84X,ETORO,2549
65,ETFEXSADE0002635307X,ETORO,8031
66,ETFSWDAIE00B4L5Y983X,ETORO,3040


In [17]:
fdb.queries.ingest_identifiers(ETF_ids_long, commit=True)

INFO     fin_db.queries.execute: Executed write query from write_identifiers.sql (68 rows affected).
INFO     fin_db.queries.execute: Transaction committed.


## Ingesting currencies

In [3]:
curr = pd.read_csv('../../data/currency_history.csv', index_col=0)

pair = {       #inUSD    #USDin
               # bid     # ask
    'EURUSD': ["EUR=", "USDEUR=R"],
    'GBPUSD': ["GBP=", "USDGBP=R"],
    'AUDUSD': ["AUD=", "USDAUD=R"],
    'DKKUSD': ["DKKUSD=R", "DKK="],
    'CHFUSD': ["CHFUSD=R", "CHF="],
    'CADUSD': ["CADUSD=R", "CAD="],
    'TWDUSD': ["TWDUSD=R", "TWD="],
    'JPYUSD': ["JPYUSD=R", "JPY="],
    'HKDUSD': ["HKDUSD=R", "HKD="],
    'CNYUSD': ["CNYUSD=R", "CNY="]
}

ids = {
    'internal_ticker': list(pair.keys()),
    'LSEG': [i[0] for i in pair.values()],
    'name': [
        'Euro', 'British Pound', 'Australian Dollar', 'Danish Krone',
        'Swiss Franc', 'Canadian Dollar', 'New Taiwan Dollar', 'Japanese Yen',
        'Hong Kong Dollar', 'Chinese Yuan'
    ],
    'instrument_id': [
        fdb.create_instrument_id(
            asset_class='currency',
            code=code[:3],
            hash_on=code[3:]
        ) 
        for code in pair.keys()
    ]
}

id_df = pd.DataFrame(ids)
id_df

,internal_ticker,LSEG,name,instrument_id
0,EURUSD,EUR=,Euro,CUREURXUSDXXXXXXXXXX
1,GBPUSD,GBP=,British Pound,CURGBPXUSDXXXXXXXXXX
2,AUDUSD,AUD=,Australian Dollar,CURAUDXUSDXXXXXXXXXX
3,DKKUSD,DKKUSD=R,Danish Krone,CURDKKXUSDXXXXXXXXXX
4,CHFUSD,CHFUSD=R,Swiss Franc,CURCHFXUSDXXXXXXXXXX
5,CADUSD,CADUSD=R,Canadian Dollar,CURCADXUSDXXXXXXXXXX
6,TWDUSD,TWDUSD=R,New Taiwan Dollar,CURTWDXUSDXXXXXXXXXX
7,JPYUSD,JPYUSD=R,Japanese Yen,CURJPYXUSDXXXXXXXXXX
8,HKDUSD,HKDUSD=R,Hong Kong Dollar,CURHKDXUSDXXXXXXXXXX
9,CNYUSD,CNYUSD=R,Chinese Yuan,CURCNYXUSDXXXXXXXXXX


In [29]:
instruments = id_df[['instrument_id', 'name', 'internal_ticker']].copy()
instruments['asset_class'] = 'currency'
instruments['unit'] = 'USD'

instruments.head(1)

,instrument_id,name,internal_ticker,asset_class,unit
0,CUREURXUSDXXXXXXXXXX,Euro,EURUSD,currency,USD


In [ ]:
fdb.open_session('fin_db_app')
#fdb.queries.ingest_instruments(instruments, commit=True)

INFO     fin_db.session: Database session opened successfully.
INFO     fin_db.queries.execute: Executed write query from write_instruments.sql (10 rows affected).
INFO     fin_db.queries.execute: Transaction committed.


Build midpoint for currencies

In [20]:
price_series = dict()
for k, v in pair.items():
    temp = curr[v].copy()
    # Invert the "inUSD" column to get "USDin" (effectively ask)
    temp.columns = ['bid', 'ask']
    temp['ask'] = 1 / temp['ask']
    # Priority: 1 mid 2 bid 3 ask
    temp['mid'] = temp.mean(axis=1)
    price_series[k] = (
        temp['mid'].combine_first(temp['bid']).combine_first(temp['ask'])
        .dropna()
    )

price_df = pd.DataFrame(price_series)

map = id_df.set_index('internal_ticker')['instrument_id'].to_dict()
map['Date'] = 'date'
price_df = price_df.reset_index().rename(columns=map)
print(price_df.columns)
hist_cur = price_df.melt(id_vars='date', var_name='instrument_id', value_name='close')
hist_cur['source'] = 'LSEG'
hist_cur['field'] = 'close'
hist_cur['scale'] = 0.00001
hist_cur['value'] = hist_cur['close'] / 0.00001
hist_cur = hist_cur[['instrument_id', 'field', 'date', 'source', 'value', 'scale']].dropna()


changes = price_df.set_index('date').pct_change().reset_index()

changes

Index(['date', 'CUREURXUSDXXXXXXXXXX', 'CURGBPXUSDXXXXXXXXXX',
       'CURAUDXUSDXXXXXXXXXX', 'CURDKKXUSDXXXXXXXXXX', 'CURCHFXUSDXXXXXXXXXX',
       'CURCADXUSDXXXXXXXXXX', 'CURTWDXUSDXXXXXXXXXX', 'CURJPYXUSDXXXXXXXXXX',
       'CURHKDXUSDXXXXXXXXXX', 'CURCNYXUSDXXXXXXXXXX'],
      dtype='object')


/var/folders/j8/sg62rnd55jn7201xgrcdz18w0000gn/T/ipykernel_60100/4228214440.py:28:FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.


,date,CUREURXUSDXXXXXXXXXX,CURGBPXUSDXXXXXXXXXX,CURAUDXUSDXXXXXXXXXX,CURDKKXUSDXXXXXXXXXX,CURCHFXUSDXXXXXXXXXX,CURCADXUSDXXXXXXXXXX,CURTWDXUSDXXXXXXXXXX,CURJPYXUSDXXXXXXXXXX,CURHKDXUSDXXXXXXXXXX,CURCNYXUSDXXXXXXXXXX
0,1979-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1980-01-02,0.004527,0.003562,0.001745,0.005896,0.010127,0.002230,NaN,0.007758,0.005584,NaN
2,1980-01-03,0.001835,0.004729,0.001343,0.002627,0.004961,-0.003845,NaN,0.000420,-0.001116,NaN
3,1980-01-04,-0.003170,-0.002925,0.000101,-0.004447,-0.005063,0.001369,NaN,0.015119,0.001523,NaN
4,1980-01-07,0.002578,0.010633,0.003337,0.002153,0.004258,-0.000599,NaN,0.014036,-0.000710,NaN
...,...,...,...,...,...,...,...,...,...,...,...
12049,2026-03-16,0.007723,0.007069,0.013061,0.007791,0.004566,0.002335,0.004121,0.004273,-0.000332,0.000068
12050,2026-03-17,0.003022,0.002703,0.004644,0.002812,0.003803,-0.000319,0.003121,0.000254,-0.000738,0.001351
12051,2026-03-18,-0.007593,-0.007338,-0.011324,-0.007695,-0.010636,-0.002999,-0.004781,-0.005381,-0.000274,0.002030
12052,2026-03-19,0.011995,0.013125,0.008916,0.012251,0.006482,-0.000493,0.005297,0.013567,0.000597,-0.003995


In [ ]:
fdb.open_session('fin_db_app')
#fdb.queries.ingest_observations(hist_cur, commit=True)

INFO     fin_db.session: Database session opened successfully.
INFO     fin_db.queries.execute: Executed write query from write_observations.sql (118457 rows affected).
INFO     fin_db.queries.execute: Transaction committed.
INFO     fin_db.queries.execute: Executed write query from log_updates.sql (0 rows affected).
INFO     fin_db.queries.execute: Transaction committed.


In [33]:
id_df = pd.DataFrame(ids['instrument_id'], ids['LSEG']).reset_index()
id_df.columns = ['ext_id', 'instrument_id']
id_df['source'] = 'LSEG'

id_df.head(2)

,ext_id,instrument_id,source
0,EUR=,CUREURXUSDXXXXXXXXXX,LSEG
1,GBP=,CURGBPXUSDXXXXXXXXXX,LSEG


In [ ]:
# fdb.queries.ingest_identifiers(id_df, commit=True)

INFO     fin_db.queries.execute: Executed write query from write_identifiers.sql (10 rows affected).
INFO     fin_db.queries.execute: Transaction committed.
